# Summarization Module (The Scribe)

This notebook creates the final product: a **Structured Summary** from the raw entities we extracted.

We use a **Template-Based Generation** approach (Extractive Summarization) to ensure accuracy. The goal is to produce a "SOAP Note" style output.

In [1]:
import pandas as pd
import ast
import os

# Settings
pd.set_option('display.max_colwidth', None)

# Paths
INPUT_FILE = "../data/processed/tagged_notes.csv"
OUTPUT_FILE = "../data/processed/final_summaries.csv"

## 1. Load Extracted Data

In [2]:
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Input file {INPUT_FILE} not found. Did you run 02_extraction.ipynb?")

df = pd.read_csv(INPUT_FILE)

# Convert stringified lists back to python lists
cols_to_parse = ['problems', 'treatments', 'tests']
for col in cols_to_parse:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

print(f"Loaded {len(df)} records.")
df.head(2)

Loaded 100 records.


,medical_specialty,transcription,problems,treatments,tests
0,Allergy / Immunology,"SUBJECTIVE:, This 23-year-old white female presents with complaint of allergies. She used to have allergies when she lived in Seattle but she thinks they are worse here. In the past, she has tried Claritin, and Zyrtec. Both worked for short time but then seemed to lose effectiveness. She has used Allegra also. She used that last summer and she began using it again two weeks ago. It does not appear to be working very well. She has used over-the-counter sprays but no prescription nasal sprays. She does have asthma but doest not require daily medication for this and does not think it is flaring up.,MEDICATIONS: , Her only medication currently is Ortho Tri-Cyclen and the Allegra.,ALLERGIES: , She has no known medicine allergies.,OBJECTIVE:,Vitals: Weight was 130 pounds and blood pressure 124/78.,HEENT: Her throat was mildly erythematous without exudate. Nasal mucosa was erythematous and swollen. Only clear drainage was seen. TMs were clear.,Neck: Supple without adenopathy.,Lungs: Clear.,ASSESSMENT:, Allergic rhinitis.,PLAN:,1. She will try Zyrtec instead of Allegra again. Another option will be to use loratadine. She does not think she has prescription coverage so that might be cheaper.,2. Samples of Nasonex two sprays in each nostril given for three weeks. A prescription was written as well.",[],[medication],[]
1,Bariatrics,"PAST MEDICAL HISTORY:, He has difficulty climbing stairs, difficulty with airline seats, tying shoes, used to public seating, and lifting objects off the floor. He exercises three times a week at home and does cardio. He has difficulty walking two blocks or five flights of stairs. Difficulty with snoring. He has muscle and joint pains including knee pain, back pain, foot and ankle pain, and swelling. He has gastroesophageal reflux disease.,PAST SURGICAL HISTORY:, Includes reconstructive surgery on his right hand 13 years ago. ,SOCIAL HISTORY:, He is currently single. He has about ten drinks a year. He had smoked significantly up until several months ago. He now smokes less than three cigarettes a day.,FAMILY HISTORY:, Heart disease in both grandfathers, grandmother with stroke, and a grandmother with diabetes. Denies obesity and hypertension in other family members.,CURRENT MEDICATIONS:, None.,ALLERGIES:, He is allergic to Penicillin.,MISCELLANEOUS/EATING HISTORY:, He has been going to support groups for seven months with Lynn Holmberg in Greenwich and he is from Eastchester, New York and he feels that we are the appropriate program. He had a poor experience with the Greenwich program. Eating history, he is not an emotional eater. Does not like sweets. He likes big portions and carbohydrates. He likes chicken and not steak. He currently weighs 312 pounds. Ideal body weight would be 170 pounds. He is 142 pounds overweight. If ,he lost 60% of his excess body weight that would be 84 pounds and he should weigh about 228.,REVIEW OF SYSTEMS: ,Negative for head, neck, heart, lungs, GI, GU, orthopedic, and skin. Specifically denies chest pain, heart attack, coronary artery disease, congestive heart failure, arrhythmia, atrial fibrillation, pacemaker, high cholesterol, pulmonary embolism, high blood pressure, CVA, venous insufficiency, thrombophlebitis, asthma, shortness of breath, COPD, emphysema, sleep apnea, diabetes, leg and foot swelling, osteoarthritis, rheumatoid arthritis, hiatal hernia, peptic ulcer disease, gallstones, infected gallbladder, pancreatitis, fatty liver, hepatitis, hemorrhoids, rectal bleeding, polyps, incontinence of stool, urinary stress incontinence, or cancer. Denies cellulitis, pseudotumor cerebri, meningitis, or encephalitis.,PHYSICAL EXAMINATION:, He is alert and oriented x 3. Cranial nerves II-XII are intact. Afebrile. Vital Signs are stable.","[hypertension, diabetes, disease, fibrillation, pain]",[surgery],[]


## 2. Define Scribe Template (SOAP)
We map our extracted entities to the standard medical note format:
*   **S (Subjective)**: The patient's story (Raw Text summary - simplistic for now).
*   **O (Objective)**: Tests and Findings.
*   **A (Assessment)**: Diagnoses/Problems identified.
*   **P (Plan)**: Treatments and Medications.

In [3]:
def generate_soap_note(row):
    # Helpers to format lists
    def fmt(lst):
        return ", ".join(lst) if lst else "None detected"

    problems = fmt(row['problems'])
    treatments = fmt(row['treatments'])
    tests = fmt(row['tests'])
    
    # Construct the summary
    summary = (
        f"**SUBJECTIVE / HISTORY**:\n"
        f"Patient presented for {row['medical_specialty']}. "
        f"(See full transcription for details)\n\n"
        
        f"**OBJECTIVE / TESTS**:\n"
        f"{tests}\n\n"
        
        f"**ASSESSMENT / PROBLEMS**:\n"
        f"{problems}\n\n"
        
        f"**PLAN / TREATMENTS**:\n"
        f"{treatments}"
    )
    return summary

df['generated_summary'] = df.apply(generate_soap_note, axis=1)

## 3. Review Results (Scribe Check)
Comparing the raw text with our generated summary.

In [4]:
def print_comparison(idx):
    print("="*80)
    print(f"NOTE ID: {idx} | SPECIALTY: {df.loc[idx, 'medical_specialty']}")
    print("="*80)
    print("--- ORIGINAL TRANSCRIPTION ---")
    print(str(df.loc[idx, 'transcription'])[:500] + "... [truncated]")
    print("\n--- GENERATED SOAP NOTE ---")
    print(df.loc[idx, 'generated_summary'])
    print("\n")

# Show a few examples
for i in range(3):
    if i < len(df):
        print_comparison(i)

NOTE ID: 0 | SPECIALTY:  Allergy / Immunology
--- ORIGINAL TRANSCRIPTION ---
SUBJECTIVE:,  This 23-year-old white female presents with complaint of allergies.  She used to have allergies when she lived in Seattle but she thinks they are worse here.  In the past, she has tried Claritin, and Zyrtec.  Both worked for short time but then seemed to lose effectiveness.  She has used Allegra also.  She used that last summer and she began using it again two weeks ago.  It does not appear to be working very well.  She has used over-the-counter sprays but no prescription nasal spr... [truncated]

--- GENERATED SOAP NOTE ---
**SUBJECTIVE / HISTORY**:
Patient presented for  Allergy / Immunology. (See full transcription for details)

**OBJECTIVE / TESTS**:
None detected

**ASSESSMENT / PROBLEMS**:
None detected

**PLAN / TREATMENTS**:
medication


NOTE ID: 1 | SPECIALTY:  Bariatrics
--- ORIGINAL TRANSCRIPTION ---
PAST MEDICAL HISTORY:, He has difficulty climbing stairs, difficulty with airline seat

## 4. Save Final Output

In [5]:
cols_to_save = ['medical_specialty', 'generated_summary', 'transcription']
df[cols_to_save].to_csv(OUTPUT_FILE, index=False)
print(f"Final summaries saved to {OUTPUT_FILE}")

Final summaries saved to ../data/processed/final_summaries.csv
